In [3]:

import json
from openai import OpenAI
from typing import Literal
from pydantic import BaseModel, Field

from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")


def initialize_index():
    reader = GithubRepositoryDataReader(
        repo_owner="evidentlyai",
        repo_name="docs",
        allowed_extensions={"md", "mdx"},
    )
    files = reader.read()

    parsed_docs = [doc.parse() for doc in files]
    chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

    index = Index(
        text_fields=["title", "description", "content"],
        keyword_fields=["filename"]
    )
    index.fit(chunked_docs)

    print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")
    return index


RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


class RAG:

    def __init__(self,
        index,
        llm_client,
        output_type = RAGResponse,
        rag_instructions = RAG_INSTRUCTIONS,
        model_name='gpt-4o-mini'
    ):
        self.index = index
        self.llm_client = llm_client

        self.output_type = output_type
        self.rag_instructions = rag_instructions
        self.model_name = model_name

    def search(self, query):
        results = self.index.search(
            query=query,
            num_results=5
        )
        return results

    def build_prompt(self, question, search_results):
        context = json.dumps(search_results, indent=2)
        return RAG_PROMPT_TEMPLATE.format(
            question=question,
            context=context
        )

    def llm(self, user_prompt):
        messages = [
            {"role": "system", "content": self.rag_instructions},
            {"role": "user", "content": user_prompt}
        ]

        response = self.llm_client.responses.parse(
            model=self.model_name,
            input=messages,
            text_format=self.output_type
        )

        return response.output_parsed

    def rag(self, question):
        search_results = self.search(question)
        prompt = self.build_prompt(question, search_results)
        answer = self.llm(prompt)
        return answer

In [4]:

openai_client = OpenAI()

index = initialize_index()

rag_instance = RAG(
    index=index,
    llm_client=openai_client,
    model_name="gpt-4o-mini",
)


response = rag_instance.rag("llm as a jury")
print(response.answer)

Indexed 385 chunks from 95 documents
## Evaluation of LLM Outputs Using Multiple LLMs

The **LLM-as-a-jury** approach utilizes multiple Large Language Models (LLMs) to evaluate the same output. This method can aggregate evaluations, declaring an output a "pass" only if all or the majority of LLMs approve, or it can explicitly highlight disagreements among them.

### Key Steps in Implementing LLM as a Jury:

1. **Setup**:
   - Install the required packages:
     ```python
     pip install evidently litellm
     ```
   - Import necessary components and set up API keys for LLM judges.

2. **Define Evaluation Criteria**:
   - Create a prompt for the LLMs that specifies the evaluation criteria for judging outputs, like appropriate tone and content in email communication.

3. **Create a Panel of Judges**:
   - Instantiate evaluators from various LLM providers using a unified evaluation prompt. Each judge assesses the tone of generated emails and produces a summary that indicates whether the 

In [5]:
print(f"Found answer: {response.found_answer}")
print(f"Confidence: {response.confidence}")
print(f"Answer type: {response.answer_type}")
print(f"Follow-up questions: {response.followup_questions}")

Found answer: True
Confidence: 0.9
Answer type: how-to
Follow-up questions: ['What are the benefits of using multiple LLMs for evaluation?', 'Can I use self-hosted LLMs in this framework?', 'How do I interpret disagreements among LLM judges?']
